# DS 4300: Spark ETL into a Medallion Lakehouse

OneShop's lakehouse is empty. In this lab you hydrate it: PySpark jobs copy OneShop's two source systems into **bronze** Iceberg tables, then clean, validate, and enrich them into **silver** tables.

| Source system | AWS service | Bronze table(s) |
| --- | --- | --- |
| OLTP database: `users`, `items`, `purchases` | **Amazon RDS for PostgreSQL** (`oneshop-db`) | `bronze.users`, `bronze.items`, `bronze.purchases` |
| Website page-view events (JSON Lines files) | **Amazon S3** (`s3://oneshop-pageviews`) | `bronze.pageviews` |

**How to use this notebook.** Run the cells top to bottom. A heading marked **(your turn)** is followed by an empty code cell: write the code that meets the goal, run it, then move on. Later steps depend on 3.4 and 6.1, so do those before continuing.

The notebook is safe to rerun: the `CREATE` statements use `IF NOT EXISTS`, and the jobs replace their tables' contents.

## Where things come from

A few names in this notebook are not plain Python. This table says what each one is and where it is defined, so nothing appears out of nowhere.

| You will see | What it does | Where it comes from |
| --- | --- | --- |
| `!aws ...`, `!spark-submit ...` | A line starting with `!` runs as a shell command in the `spark-iceberg` container, the same as typing it in `docker exec -it spark-iceberg bash`. | A Jupyter (IPython) feature. `{name}` inside the command inserts a Python variable. |
| `spark` | A ready-made Spark session, already connected to the Glue catalog. You never create it. | Made when the kernel starts: this container launches Jupyter through PySpark, which runs PySpark's `shell.py` in every kernel. Its settings come from `spark/spark-defaults.conf`. |
| `%%sql` | When it is the **first line** of a cell, the rest of the cell runs as **one** Spark SQL statement, and the result shows as a table. | Not built into Jupyter. This repo defines it in `spark/ipython-startup.py`, and all it does is `spark.sql(cell).toPandas()`. |
| `pg(query)` | Runs a SQL query **inside the Postgres source database** and returns a table. | Defined in this notebook, in Step 1. |
| `athena(sql)` | Runs a SQL query in **Athena** and returns a table. | Defined in this notebook, in Step 9 (the same function as in the Part 2 lab). |
| `boto3` | The AWS SDK for Python. | A Python package installed in the image. It reaches Floci, not real AWS, because the container sets `AWS_ENDPOINT_URL=http://floci:4566` and the credentials. |
| `pyiceberg` | Command-line tool for Iceberg catalogs. | A Python package installed in the image; the `glue` catalog is configured in `spark/.pyiceberg.yaml`. |
| `/home/iceberg/jobs/*.py` | The three PySpark ETL jobs. | The `jobs/` folder on your laptop, mounted into the container by `compose-spark-intro.yaml`. Open them in your editor to read them. |

**`%%sql` and `pg()` look alike but talk to different systems.** `%%sql` asks Spark, which only sees the lakehouse: Iceberg tables in the Glue catalog (`bronze.*`, `silver.*`). `pg()` asks Postgres, which only sees the source tables (`users`, `items`, `purchases`). Before Step 5 the data exists only in Postgres, so Step 1 needs `pg()`. After Step 5 the data is in the lakehouse, and the rest of the notebook uses `%%sql`.

## Step 0: Start the source systems

Run this on your laptop, from the repo root. Floci keeps running; `spark-iceberg` is rebuilt, and the one-shot `loadgen` container creates the RDS database and the page-view bucket and fills them with mock data.

```bash
docker compose -f docker-compose.yaml -f compose-spark-intro.yaml up -d --build
docker logs -f loadgen        # wait for: [loadgen] done   (then Ctrl-C)
```

Then open this notebook. If it was already open, restart the kernel (**Kernel ➤ Restart Kernel**).

## Step 1: Look at the source systems

### Amazon RDS for PostgreSQL

`loadgen` created the database with `aws rds create-db-instance`. Floci runs a real Postgres 16 server behind it.

In [ ]:
!aws rds describe-db-instances --query 'DBInstances[].{id:DBInstanceIdentifier,engine:Engine,status:DBInstanceStatus,address:Endpoint.Address,port:Endpoint.Port}' --output table

### The `pg()` helper

`%%sql` cannot see Postgres (see "Where things come from"), so the next cell defines a small helper, `pg(query)`, that runs a query inside Postgres:

1. **Find the database.** Ask the RDS API for `oneshop-db`'s address and port and build a JDBC URL (`jdbc:postgresql://host:port/oneshop`). JDBC is Java's standard way to connect to a database, and Spark runs on Java.
2. **Log in as `etluser`**, the read-only account the ETL jobs use.
3. **Run the query.** `spark.read.jdbc` expects a table name, so `pg()` wraps your query as `(query) AS q`. Postgres runs the whole query and Spark receives only the result rows.
4. **Return a pandas DataFrame** (`.toPandas()`), which Jupyter shows as a table.

The real loader in Step 5 uses the same Spark JDBC reader, so this is also a preview of how it reads.

In [ ]:
import boto3

endpoint = boto3.client("rds").describe_db_instances(DBInstanceIdentifier="oneshop-db")["DBInstances"][0]["Endpoint"]
JDBC_URL = f"jdbc:postgresql://{endpoint['Address']}:{endpoint['Port']}/oneshop"
ETL_LOGIN = {"user": "etluser", "password": "etlpassword", "driver": "org.postgresql.Driver"}


def pg(query):
    """Run a SQL query inside the Postgres source database; return the rows as a pandas DataFrame."""
    return spark.read.jdbc(JDBC_URL, f"({query}) AS q", properties=ETL_LOGIN).toPandas()


JDBC_URL

In [ ]:
pg("""SELECT 'users' AS source_table, count(*) AS row_count FROM users
UNION ALL SELECT 'items', count(*) FROM items
UNION ALL SELECT 'purchases', count(*) FROM purchases""")

In [ ]:
pg("SELECT * FROM purchases ORDER BY id LIMIT 5")

### 1.1 Purchases per day (your turn)

**Goal:** with `pg()`, count the purchases per day in Postgres, oldest day first. In Postgres, `created_at::date` turns a timestamp into a date. Expect about a week of days.

### Least privilege

`etluser` may only read. This cell tries to delete every purchase as `etluser` and shows that Postgres refuses, so a buggy or leaked ETL job cannot damage the source system.

It uses `spark._jvm`, a handle into the Java process Spark runs in, to open a plain JDBC connection with the same driver Spark uses. You will not need `spark._jvm` anywhere else.

In [ ]:
conn = spark._jvm.java.sql.DriverManager.getConnection(JDBC_URL, ETL_LOGIN["user"], ETL_LOGIN["password"])
try:
    conn.createStatement().executeUpdate("DELETE FROM purchases")
except Exception as e:
    print(next(line for line in str(e).splitlines() if "permission denied" in line))
finally:
    conn.close()

### Amazon S3: page-view events

The website writes one JSON Lines object per hour under `yyyy/mm/dd/hh/` (the layout Kinesis Data Firehose uses). JSON Lines means one JSON object per line. These are plain files, not an Iceberg table.

In [ ]:
!aws s3 ls s3://oneshop-pageviews/ --recursive --summarize | tail -4

In [ ]:
# boto3 (the AWS SDK) reads the first object and prints three events.
s3 = boto3.client("s3")
first_key = s3.list_objects_v2(Bucket="oneshop-pageviews", MaxKeys=1)["Contents"][0]["Key"]
body = s3.get_object(Bucket="oneshop-pageviews", Key=first_key)["Body"].read().decode()
print(first_key)
print("\n".join(body.splitlines()[:3]))

## Step 2: One namespace per Medallion layer

| Layer | Holds |
| --- | --- |
| **bronze** | Raw copies of the sources, 1:1, typed for Iceberg |
| **silver** | Cleaned, validated, and enriched data |
| **gold** | Business aggregates (a later lab) |

In Spark, `CREATE DATABASE` makes an Iceberg namespace, which Glue stores as a database. **Each one needs a `LOCATION`**: Floci's Glue refuses to create a table in a database without one.

From here on, each `%%sql` cell holds exactly one statement.

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS bronze LOCATION 's3://warehouse/bronze'

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS silver LOCATION 's3://warehouse/silver'

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS gold LOCATION 's3://warehouse/gold'

In [ ]:
!aws glue get-databases --query 'DatabaseList[].[Name,LocationUri]' --output table

## Step 3: Bronze tables

Bronze tables copy their source 1:1, with Iceberg types (Postgres `SERIAL` becomes `BIGINT`, `VARCHAR` becomes `STRING`).

- `PARTITIONED BY (days(created_at))` is **hidden partitioning**: Iceberg groups the files by day without adding a column, and queries that filter on `created_at` skip the days they don't need.
- `'format-version' = '2'` enables row-level updates and deletes.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS bronze.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Dimension table for user information')

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS bronze.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Dimension table for product items')

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS bronze.purchases (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Fact table for purchase transactions')

### 3.4 Model the page views (your turn)

**Goal:** create `bronze.pageviews` to hold the JSON events you printed in Step 1, one column per JSON field, with these names and types: `user_id BIGINT`, `url STRING`, `channel STRING`, `received_at TIMESTAMP`. Partition it by the **day** of `received_at`, and make it format version 2. Use the three tables above as a pattern.

The loader in Step 6 writes into this table by column name, so the names must match exactly.

## Step 4: Silver tables

Silver tables add value on top of bronze:

- **derived columns**: `valid_email`, `full_name`, `total_price`, `purchase_date`, `purchase_hour`
- **denormalized columns** copied in from other tables (`user_email`, `item_name`, `item_category`), so analysts don't have to join

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS silver.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    valid_email BOOLEAN,
    full_name STRING
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Validated dimension table for user information')

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS silver.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7,2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (category)
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Validated dimension table for product items')

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS silver.purchases_enriched (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12,2),
    total_price DECIMAL(14,2),
    user_email STRING,
    item_name STRING,
    item_category STRING,
    purchase_date DATE,
    purchase_hour INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(created_at))
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Validated and enriched fact table for purchase transactions')

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS silver.pageviews_by_items (
    user_id BIGINT,
    item_id BIGINT,
    page STRING,
    item_name STRING,
    item_category STRING,
    channel STRING,
    received_at TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(received_at))
TBLPROPERTIES ('format-version' = '2', 'comment' = 'Product page views enriched with item details')

### Check the tables

`SHOW TABLES`, `DESCRIBE TABLE`, and `SHOW TBLPROPERTIES` work here. Spark's metadata tables (such as `bronze.users.properties`) do not work against Floci.

In [ ]:
%%sql
SHOW TABLES IN bronze

In [ ]:
%%sql
SHOW TABLES IN silver

In [ ]:
%%sql
DESCRIBE TABLE bronze.users

In [ ]:
%%sql
SHOW TBLPROPERTIES silver.purchases_enriched

## Step 5: Load bronze from RDS Postgres

`jobs/postgres_loader.py` reads each Postgres table over JDBC as `etluser`, casts the columns to the bronze types, and replaces the bronze table's contents. Open it in your editor to see the code.

`spark-submit` runs a PySpark script as its own Spark application, the way a scheduler runs jobs in production. It is Spark's command-line launcher, installed in the container. The PostgreSQL driver and the S3 connector the jobs need are already in the image, so no extra options are needed.

The notebook's own Spark session already uses the Spark UI port 4040, so each job prints a warning that it moved to the next free port (4041, 4042, ...). That warning is harmless.

In [ ]:
!spark-submit /home/iceberg/jobs/postgres_loader.py

In [ ]:
%%sql
SELECT 'users' AS bronze_table, count(*) AS row_count FROM bronze.users
UNION ALL SELECT 'items', count(*) FROM bronze.items
UNION ALL SELECT 'purchases', count(*) FROM bronze.purchases

Hidden partitioning on disk: one `created_at_day=` folder per day, but no `created_at_day` column in the table.

In [ ]:
!aws s3 ls s3://warehouse/bronze/purchases/data/

## Step 6: Load bronze from the S3 page views

`jobs/pageviews_loader.py` reads the JSON Lines files through Hadoop's S3A connector (paths start with `s3a://`). It declares the schema, so `received_at` is read directly as a timestamp, then replaces the contents of `bronze.pageviews`.

### 6.1 Run the page-view loader (your turn)

**Goal:** run `/home/iceberg/jobs/pageviews_loader.py` the same way Step 5 ran the Postgres loader. Expect `bronze.pageviews: 20000 rows`. If it fails with a missing column, compare your table from 3.4 with the column names given there.

### 6.2 Views per channel (your turn)

**Goal:** with `%%sql`, count the rows in `bronze.pageviews` per `channel`, most views first.

## Step 7: Bronze to silver

`jobs/bronze_to_silver.py` fixes the problems in the source data and enriches the fact tables:

| Silver table | What the job does |
| --- | --- |
| `silver.users` | `valid_email` is `false` for missing or malformed emails; `full_name` = first + last name |
| `silver.items` | negative prices become 0; categories are upper-cased |
| `silver.purchases_enriched` | joins purchases with users and items; adds `total_price = quantity * purchase_price`, `purchase_date`, `purchase_hour` |
| `silver.pageviews_by_items` | splits `/products/42` into `page` and `item_id`; drops views of other pages (`/`, `/cart`, ...); joins with items |

Silver writes use `overwritePartitions()`, which replaces only the partitions present in this run.

In [ ]:
!spark-submit /home/iceberg/jobs/bronze_to_silver.py

## Step 8: Check the silver layer (your turn)

Use `%%sql`, one statement per cell. Add cells if you need more.

### 8.1 Email validation

**Goal:** count the users in `silver.users` by `valid_email`. Then, in the second cell, show five users whose email is not valid (`id`, `full_name`, `email`). Which kinds of bad email does the check catch?

### 8.2 Price and category cleanup

**Goal:** list every item with a negative price in `bronze.items`, showing its bronze and silver `category` and `price` side by side. Join `bronze.items` and `silver.items` on `id`.

### 8.3 Enriched purchases

**Goal:** show the five most recent rows of `silver.purchases_enriched` (by `created_at`) with `user_email`, `item_name`, `item_category`, `quantity`, `purchase_price`, `total_price`, `purchase_date`, and `purchase_hour`. Check that `total_price` is `quantity * purchase_price`. Why might `user_email` be empty for some rows?

### 8.4 Where did the page views go?

**Goal:** in one query, show three numbers: the rows in `bronze.pageviews`, the bronze rows whose `url` is **not** a product page (`url NOT LIKE '/products/%'`), and the rows in `silver.pageviews_by_items`. Explain the difference.

Hint: `SELECT (SELECT count(*) FROM ...) AS a, (SELECT count(*) FROM ...) AS b` is a single statement.

### 8.5 The same table, seen by PyIceberg

**Goal:** use the `pyiceberg` command-line tool (with `!`) to list the tables in the `silver` namespace of the `glue` catalog, then describe `silver.purchases_enriched`. Find its partition spec and its snapshots.

Hint: `!pyiceberg --catalog glue list silver`. For `describe`, add `--entity=table`: without it the tool first tries the name as a namespace, and Floci answers that with an error.

## Step 9: Query silver with Athena

The silver tables are registered in Glue, so Athena can query them with no extra setup.

The next cell defines `athena(sql)`, the same helper as in the Part 2 lab. It makes three boto3 calls: start the query, check its status until it finishes, then fetch the rows. It returns a pandas DataFrame in which every value is text. Its `database` argument defaults to `silver`, so table names need no prefix.

In [ ]:
import time

import pandas as pd

athena_client = boto3.client("athena")


def athena(sql, database="silver"):
    """Start a query, wait for it to finish, and return its rows as a DataFrame."""
    qid = athena_client.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
    )["QueryExecutionId"]
    while True:
        status = athena_client.get_query_execution(QueryExecutionId=qid)["QueryExecution"]["Status"]
        if status["State"] not in ("QUEUED", "RUNNING"):
            break
        time.sleep(0.5)
    if status["State"] != "SUCCEEDED":
        raise RuntimeError(f"{status['State']}: {status.get('StateChangeReason')}")
    rows = athena_client.get_query_results(QueryExecutionId=qid)["ResultSet"]["Rows"]
    values = [[col.get("VarCharValue") for col in row["Data"]] for row in rows]
    return pd.DataFrame(values[1:], columns=values[0]) if values else pd.DataFrame()

### 9.1 Revenue by category (your turn)

**Goal:** with `athena()`, compute the total revenue (`sum(total_price)`) and the number of purchases for each `item_category` in `purchases_enriched`, highest revenue first. This is the kind of question a gold table answers in the next lab.

## Shutting down

- Rerunning this notebook or any job is always safe.
- `docker compose -f docker-compose.yaml -f compose-spark-intro.yaml stop`, and later `... start`, keeps all the data.